# conda-express in the Browser

**conda-express** is a lightweight, Rust-based tool that bootstraps a full conda environment entirely inside your browser using WebAssembly (WASM).

This demo runs a real Python 3.13 kernel (via [xeus-python](https://github.com/jupyter-xeus/xeus-python)) compiled to WebAssembly, with packages pre-installed from [emscripten-forge](https://github.com/emscripten-forge/recipes).

---

## Architecture

```
Browser
  └── JupyterLite (main thread)
       └── xeus-python kernel (WebWorker)
            └── Python 3.13 (WASM/Emscripten)
                 └── numpy, matplotlib, ipywidgets
```

All packages were installed **at build time** from `emscripten-forge-4x` by [jupyterlite-xeus](https://github.com/jupyterlite/xeus). No server needed at runtime — everything runs in your browser.


## 1. Verify the Environment

In [ ]:
import sys
import platform

print(f"Python {sys.version}")
print(f"Platform: {sys.platform}")
print(f"Machine:  {platform.machine()}")

# Emscripten builds expose runtime info on sys
if hasattr(sys, '_emscripten_info'):
    info = sys._emscripten_info
    print(f"Emscripten SDK: {info.emscripten_version}")
    print(f"Running in browser: yes")
else:
    print("Not running in Emscripten")

## 2. NumPy — Scientific Computing

In [ ]:
import numpy as np

print(f"NumPy {np.__version__}")

# FFT of a sine wave — runs fully in WASM
t = np.linspace(0, 1, 1024, endpoint=False)
signal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
fft = np.fft.rfft(signal)
freqs = np.fft.rfftfreq(len(t))

peak_idx = np.argmax(np.abs(fft[1:])) + 1
print(f"Dominant frequency: {freqs[peak_idx] * 1024:.1f} Hz (expected ~5 Hz)")

## 3. Matplotlib — Visualisation

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

print(f"Matplotlib {matplotlib.__version__}")

# Lemniscate of Bernoulli (∞ symbol)
theta = np.linspace(0, 2 * np.pi, 500)
a = 2.0
r = a * np.sqrt(np.abs(np.cos(2 * theta)))
x = r * np.cos(theta)
y = r * np.sin(theta)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("conda-express — Python + WASM in the browser", fontsize=13)

# Left: FFT spectrum
axes[0].plot(freqs[:len(freqs)//4] * 1024, np.abs(fft[:len(freqs)//4]),
             color='steelblue', linewidth=1.5)
axes[0].set_title("FFT spectrum of a sine wave")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Amplitude")
axes[0].axvline(5, color='tomato', linestyle='--', label='5 Hz')
axes[0].axvline(12, color='orange', linestyle='--', label='12 Hz')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: lemniscate
axes[1].plot(x, y, color='mediumpurple', linewidth=2)
axes[1].set_aspect('equal')
axes[1].set_title("Lemniscate of Bernoulli")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. How It Works

**cx-wasm** is the Rust/WASM core of conda-express. It:

1. Fetches sharded repodata (CEP-16 format) from `emscripten-forge` using synchronous XHR inside a WebWorker.
2. Decodes `msgpack.zst` shards with pure-Rust `ruzstd` + `rmp-serde` — no native libs needed.
3. Solves dependencies in-process via the `rattler` SAT solver (resolvo).
4. Streams `.conda` / `.tar.bz2` packages directly from the CDN and extracts them into Emscripten MEMFS.

All of this runs in a **single Rust WASM call** — no intermediate JSON round-trips between Python, JavaScript, and WASM.

```
Python (CxWasmSolver)
  └── json.dumps(request)
       └── JS: cx_fetch_and_solve()
            └── Rust WASM: fetch shards → solve → return solution
                 └── JS: cx_extract_package() per package
                      └── Rust WASM: download + extract → MEMFS
```


## 5. Runtime `%cx install` via cx-wasm

If this demo was built with `pixi run -e lite lite-build-local`, then `cx-wasm-kernel`, `conda`,
and `conda-emscripten` are all pre-installed in this kernel.

The `%cx` magic is the browser equivalent of the native `cx` CLI:

```
terminal:      cx install zlib
JupyterLite:  %cx install zlib
```

### How auto-loading works

1. **Import triggers background loading** — `import cx_wasm_bridge` schedules
   WASM loading as a background asyncio task.  The ~2 MB binary starts loading
   from the kernel's in-memory filesystem right away.

2. **`await setup()` waits and registers** — Awaits the background task, then
   registers bridge functions on the JS global scope.  Near-instant if loading
   already completed.

3. **`%cx` auto-initialises** — Running `%cx install <pkg>` calls `setup()`
   automatically on first use, so no explicit setup cell is needed.


In [ ]:
try:
    import cx_wasm_bridge
    await cx_wasm_bridge.setup()
    print("cx-wasm ready — %cx install <pkg> is now available")
except ImportError:
    print("cx-wasm-kernel not installed. Rebuild with: pixi run -e lite lite-build-local")

### Install packages with `%cx`

The `%cx` magic mirrors the native `cx` CLI — same commands, same mental model:

| Terminal | JupyterLite |
|---|---|
| `cx install zlib` | `%cx install zlib` |
| `cx update numpy` | `%cx update numpy` |
| `cx list` | `%cx list` |
| `cx remove zlib` | `%cx remove zlib` |

`%cx` auto-initialises the cx-wasm bridge on first use, so no explicit
`await setup()` cell is required.


In [ ]:
%cx install zlib

In [ ]:
%cx list

## 6. What's Next

### Already shipped
- **cx-wasm WASM bridge** — Rust resolvo solver + extractor running in xeus-python kernel
- **`%cx` magic** — `%cx install <pkg>` mirrors the native `cx` CLI; auto-setup via cx_wasm_bridge
- **conda plugin hook** — `conda_pre_commands` triggers background WASM loading before any conda command
- **Full local stack** — `conda`, `conda-emscripten`, and `cx-wasm-kernel` pre-built and installable into the kernel

### Up next
- Publish `@conda-express/web` as an npm package for standalone browser embedding
- CI/CD deployment of this JupyterLite demo to GitHub Pages
- Submit `conda`, `conda-emscripten`, and `cx-wasm-kernel` packages to `emscripten-forge` so no local build is needed
